# Class 2: Optimization Problems

In [32]:
import sys
import os
import random
import time

sys.path.append(os.path.abspath(".."))

from src.utils import buildMenu, testGreedys, Food

## Binary Tree:

- The tre is buld top down starting with the root
- So in each level the number of nodes DOUBLES, (level 0: 1 node, level 1: 2 nodes, level 2, 4 nodes...), so the number of nodes at level k is: $2^{k}$
- So, if we want to know the TOTAL number of nodes in the tree we just need to sum all nodes in each level: $\sum_{i=1}^{n}$ $2^{i}$
    - If there is room for that item in the knapsack, a node is constructed that reflects the consequence of choosing to to take that item
    - We also explore the consequences of NOT TAKING THAT ITEM
- The process is then applied recursively to non-leaf children
- We chose a node with the highest value that meets the constraitns
- **IS AN OBVIOUS OPTIMIZATION: we don't need to explore the whole tree, if we get to a point where the backpack is overstuff, we just stop! we can abort early ignore leafs where we exceed the constraints.**
- But... it won't reduce the code complexity

In [16]:
names = ["wine", "beer", "pizza", "burger", "fries", "cola", "apple", "donut", "cake"]
values = [89, 90, 95, 100, 90, 79, 50, 10]
calories = [123, 154, 258, 354, 365, 150, 95, 195]

foods = buildMenu(names, values, calories)

In [17]:
def maxVal(toConsider, avail):
    """
    Assummes to Consider a ilst of items, avail is a weight
    Returns a tuple of the total vaue of a solution to 0/1 knapsack prlbem and the items of that stolution
    TO CONSDIER: those items that nodes higher up in the trree have not yet considered (corersponding to earlier calls in the recursive call stack)
    avail: the amount of space available
    """
    # if there is nothing more to evaluate or there is no space left, we can't do anything!
    if toConsider == [] or avail == 0:

        result = (0, ())

    # but if we have an object, the first step is to check if the cost of that value exceeds the availability, if it is we don't need to explore that branch!
    # so this is the case where we ignore the left branch and we pass to the right branch, that is why we continue with the same avail
    elif toConsider[0].getCost() > avail:
        result = maxVal(toConsider[1:], avail)

    ## but, if the first value does not exceed the availability we need to explore both branches, left and right
    else:
        nextItem = toConsider[0]

        # first we explore left branch (chosing the product),
        # it looks similar to the previous situation , with the exception that we are counting for the cost of the first item

        withVal, withToTake = maxVal(toConsider[1:], avail - nextItem.getCost())
        withVal += nextItem.getValue()

        # then we explore the right branch
        # but also we are repearing the situation where the cost exeeds the avail, but now because we want to compare it with the previous situation!

        withoutVal, withoutToTake = maxVal(toConsider[1:], avail)

        if withVal > withoutVal:
            result = (withVal, withToTake + (nextItem,))

        else:
            result = (withoutVal, withoutToTake)

    return result

In [18]:
def testMaxVal(foods, maxUnits, printItems=True):

    print("Use search tree to allocate", maxUnits, "calories")

    val, taken = maxVal(foods, maxUnits)

    print("Total vale of items taken =", val)

    if printItems:

        for item in taken:
            print("   ", item)

In [ ]:
%%timeit

# using the previous lecture brute force algorithm we have an average of 434 microseconds per execution
testGreedys(750, foods)

In [ ]:
%%timeit
# meanwhile the search tree algorithm only took 205 microseconds, nearly half!
# also we can get a higher result, with 353 compared to the 318 of the greddy algorithm
testMaxVal(foods, 750)

## Search tree worked great
- Gave us a better answer
- finished quickly
- but $2^{8}$ is not a large number
 - we should look at what happens when we have a more extensive meny to choose from!

In [15]:
def buildLargeMenu(numItems, maxVal, maxCost):

    items = []

    for i in range(numItems):

        # basically we are just creating different food instances with random values and calories

        items.append(
            Food(str(i), random.randint(1, maxVal), random.randint(1, maxCost))
        )

    return items

In [ ]:
# now we want to test different sizes of menus on the max val algorithm, and see how much it takes as k grows
for numItems in (5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60):
    start = time.time()
    items = buildLargeMenu(numItems, 90, 250)
    print("testing max val algorithm with ", numItems, "items \n")
    testMaxVal(items, 750, False)
    end = time.time()

    print(f"\niteration took {end - start:.4f} seconds\n ")
    print("==================================================")

    # in each iteration, we are compyuting 2 power (60) AN ENOURMOUS NUMBER

**how much patience do we have? ...**

- is an inherently expotinential problem! remember that we are growing at pace of  $2^{k}$!!
- **The solution: DYNAMIC PROGRAMMING**
- Funny fact: the name 'dynamic programming' was invented because Richard Bellman was working funded by the defense department of the US, and they didn't like the application of mathematics, and Bellman was afraid so he just inveted something that no one would know what it meant!


## Dynamic programming solution

In [27]:
## lets see how dynamic programming can help us to improve effience, first we define a common fib function
def fib(n):
    if n == 0 or n == 1:
        return 1
    else:
        return fib(n - 1) + fib(n - 2)

In [ ]:
for i in range(121):
    # these are not huge numbers but the computation acutally seems to be growing faster than the results!
    print("fib(" + str(i) + ") =", fib(i))
    # it is gonna get slower and slower at 35

**But why?**
- we are repeating the same calculations over and over again! 
- and how we avoid to do the same work over and over again... **you store the answer and then look it up when you need it!**
- it is called MEMOIZATION, craeting a table to record what we've done

In [29]:
def fastFib(n, memo={}):
    """
    Assumes n is an int >= 0, memo used only by recursive calls
    Returns Fibonacci of n
    """

    if n == 0 or n == 1:
        return 1

    try:
        return memo[n]

    except KeyError:

        result = fastFib(n - 1, memo) + fastFib(n - 2, memo)

        # now we are using a dictionary to store the the results!
        # and if we need to use calculate again, we just look up in the dictionary!
        memo[n] = result

        return result

In [ ]:
%%timeit
for i in range(121):
    # now lets try with the memoization trick, AN ENOURMOUS DIFFERENCE
    print("fib(" + str(i) + ") =", fastFib(i))

    # and if we check the time, now the loop only took 2.47 milliseconds!

## When does it work this solution?

- **Optimal substructure**: a globally optimal solution can be found by combining optimal solutions to local subproblems! 
     - for x > 1, fib(x) = fib(x-1) + fib(x-2)

- **Overlapping subproblems***: finding an optimal soltuion involves solving the same problem multiple times 
     - compute fib(x) over and over again
     - but also there will be situations where even when we have taken different decisions, that does not matter, all matters is how much calories do we have left! so if two branches have different values but the same calories left and the same food options... that is a overlapping subproblem!


- but do we have in the knacksack problem overlapping suproblems? no! , if we apply dynamic programing to this problem we will have the same result! there is not any food that is repeated.
- IF we consider another menu with several repeated foods... there will be nodes where we are solving the same problem! so we can improve the efficency

In [39]:
def fastMaxVal(toConsider, avail, memo={}):
    """
    Assumes toConsider a list of subjects, avail a weight memo supplied by recursive calls
    Returns a tuple of the totla value of a solution to the 0/1 knapsack problem and the subjects of that solution

    """
    ## the first main difference with the normal appraoch is that
    ## we start the search tree by checking the dictionary
    ## what is really interesting to me is that we are storing as the keys of the dictionary
    ## a tuple of how many values are there left in toConsider and the space avaialable
    ## we do not care about which are those items, because the list order NEVER CHANGES
    ## so we are assuming that same length = same items

    if (len(toConsider), avail) in memo:

        result = memo[(len(toConsider), avail)]

    ## then, the code seems pretty similar with the only expection that
    ## we are storing at the end in memo the result

    elif toConsider == [] or avail == 0:
        result = (0, ())

    elif toConsider[0].getCost() > avail:

        result = fastMaxVal(toConsider[1:], avail, memo)

    else:
        nextItem = toConsider[0]

        withVal, withToTake = fastMaxVal(
            toConsider[1:], avail - nextItem.getCost(), memo
        )

        withVal += nextItem.getValue()

        withoutVal, withoutToTake = fastMaxVal(toConsider[1:], avail, memo)

        if withVal > withoutVal:

            result = (withVal, withToTake + (nextItem,))
        else:

            result = (withoutVal, withoutToTake)

        # I thought, why are we just sotring the left branch result and not  the right branch
        # but in this last else we are reaching the final situation that we want to store!

        memo[len(toConsider), avail] = result

    return result

In [40]:
def testMaxVal(foods, maxUnits, algorithm, printItems=True):

    print("Menu contains", len(foods), "items")
    print("use search tree to allocate", maxUnits, "calories")

    val, taken = algorithm(foods, maxUnits)

    if printItems:
        print("Total value of items taken = ", val)

        for item in taken:
            print("  ", item)

In [ ]:
%%timeit
for numItems in (5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 65, 60):

    items = buildLargeMenu(numItems, 90, 250)

    testMaxVal(items, 750, fastMaxVal, True)

- It is really amazing how in the previous maxVal algorithm we couldn't reach the 60th iteration, and we stopped it at 23 minutes

- But now just look how we are even able to record the time with the timeit magic command!

- and if we check how much time each loop took, it is just 5.17 millliseconds!
